# 🚨 5. Auditoría de fuga de datos

Hay **fuga de datos** cuando el modelo usa, al entrenar o al validar, información que no tendría en el momento real de pronosticar. El resultado es un error de validación optimista que no se repite en producción. En un problema temporal la fuga más peligrosa es la **temporal**: usar el futuro para predecir el pasado.

Este capítulo revisa cada fuente posible de fuga y, cuando se puede, la **prueba con un experimento**. Todos los experimentos usan **solo train** (con sus folds temporales). El test no se toca.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold, cross_val_score

from comun import (estilo, cargar_panel, crear_variables, datos_modelo, folds_temporales, crear_pipeline,
                   rmse_log, RMSE_LOG, CORTE, VARIABLES_NUM, MEDIOS, AZUL, NARANJA, GRIS)

estilo()
SEED = 42
panel = cargar_panel()
datos = datos_modelo(panel)
train = datos[datos["fecha"] <= CORTE].sort_values(["fecha", "negocio"]).reset_index(drop=True)
folds = folds_temporales(train["fecha"])

# hiperparámetros de referencia (los definitivos se eligen en el cap. 8); las conclusiones no dependen de ellos
C_REF, EPS_REF = 0.1, 0.1
print(f"filas de train con historia suficiente: {len(train):,} | folds temporales: {len(folds)}")

filas de train con historia suficiente: 22,205 | folds temporales: 4


## 📋 5.1 Lista de verificación

| # | Fuente de fuga | Riesgo en este proyecto | Control | Estado |
|---|---|---|---|---|
| 1 | **Venta por medio del mismo día** como predictor | Suma exactamente el objetivo | Solo entran rezagos (participaciones de los 7 días previos) | ✅ Experimento 5.2 |
| 2 | **Variables que miran el futuro** (ventanas centradas, `shift` mal puesto) | Rezagos, medias móviles, nivel histórico | Todas las ventanas usan `shift(1)` | ✅ Experimento 5.3 |
| 3 | **Validación con filas barajadas** (k-fold aleatorio) | Días vecinos del mismo negocio en train y validación | `TimeSeriesSplit` por días, con `gap` de 7 días | ✅ Experimento 5.4 |
| 4 | **Preprocesamiento ajustado con todos los datos** (escalado, imputación) | Medias y desviaciones de test entrando al entrenamiento | Todo dentro del `Pipeline`, ajustado en cada fold | ✅ Verificación 5.5 |
| 5 | **Decisiones del EDA con datos de test** | Umbrales, agrupaciones, selección de variables | El EDA (cap. 4 y 6) usa solo train. La agrupación de medios (cap. 2) usa jul–ago | ✅ Por diseño |
| 6 | **Transformación del objetivo** | Una transformación que dependa de estadísticas de los datos | `log1p` con recorte en 0: regla fija, dentro del Pipeline | ✅ Por diseño |
| 7 | **Calendario** (día de la semana, festivo) | ¿Se conoce de antemano? | Sí: el calendario del día siguiente se conoce | ✅ Legítimo |
| 8 | **Exclusión de un negocio con datos de septiembre** | La inconsistencia del negocio con cambio de POS se vio en septiembre (test) | Decisión del negocio por consistencia de datos, no por el desempeño del modelo. Se declara | ⚠️ Declarado |
| 9 | **Día incompleto (24 sep)** | Un día a medio cargar parecería una caída real | Excluido en el cap. 2 | ✅ |
| 10 | **Elegir el corte o los hiperparámetros mirando test** | Sobreajuste al test | Corte por regla de calendario (cap. 3). Hiperparámetros con CV solo en train (cap. 8) | ✅ Por diseño |

## 🧪 5.2 Experimento: venta por medio del mismo día

Si se agregan como predictores los montos por medio de pago **del mismo día** (en log), el modelo recibe las partes cuya suma **es** el objetivo. Se compara con el modelo correcto en el último fold temporal.

In [2]:
# [IA] mismo modelo, con y sin las columnas pm_* del mismo día
for c in [f"pm_{m}" for m in MEDIOS] + ["pm_otros"]:
    train[f"hoy_{c[3:]}"] = np.log1p(train[c].clip(lower=0))
fuga = VARIABLES_NUM + [f"hoy_{m}" for m in MEDIOS] + ["hoy_otros"]

i_tr, i_val = folds[-1]
res = []
for nombre, cols in [("correcto (solo rezagos)", VARIABLES_NUM), ("con fuga (medios del mismo día)", fuga)]:
    m = crear_pipeline(C_REF, EPS_REF, num=cols).fit(train.iloc[i_tr], train["venta_total"].iloc[i_tr])
    pred = m.predict(train.iloc[i_val])
    y_val = train["venta_total"].iloc[i_val]
    z, zp = np.log1p(y_val.clip(lower=0)), np.log1p(np.clip(pred, 0, None))
    res.append((nombre, rmse_log(y_val, pred), 1 - ((z - zp) ** 2).sum() / ((z - z.mean()) ** 2).sum()))
pd.DataFrame(res, columns=["modelo", "RMSE log (validación)", "R² log (validación)"]).set_index("modelo")

,RMSE log (validación),R² log (validación)
modelo,,
correcto (solo rezagos),0.87,0.87
con fuga (medios del mismo día),0.54,0.95


### Interpretación

Con los medios del mismo día, el RMSE log cae de 0,87 a 0,54 (casi a la mitad) y el R² sube a 0,95, sin que el modelo haya aprendido nada útil para pronosticar. No llega a 1 porque en escala log la suma de las partes no es una combinación lineal de sus logaritmos. Un modelo no lineal, como un árbol, explotaría todavía más esta fuga. Es **la señal típica de fuga**: una mejora grande que viene de información que no existe al momento de pronosticar. En producción esas columnas no existen al momento de pronosticar (se conocen al cierre del día que se quiere predecir). Por eso los medios de pago **solo** entran como participación de los 7 días anteriores.

## 🧪 5.3 Experimento: ¿las variables miran el futuro?

Prueba de perturbación: se multiplican por 10 todas las ventas **después** de una fecha $d$ y se recalculan las variables. Si ninguna mira el futuro, las variables de los días $\le d+1$ deben quedar **idénticas**: el día $d+1$ usa como mucho hasta $d$. Las del día $d+2$ sí deben cambiar, porque su rezago de 1 día ya es un día alterado; eso comprueba que la prueba sí detecta cambios.

In [3]:
# [IA] prueba de perturbación sobre crear_variables
d = pd.Timestamp("2026-08-15")
alterado = panel.copy()
cols_monto = [c for c in panel if c.startswith(("pm_", "cat_"))] + ["venta_total"]
alterado.loc[alterado["fecha"] > d, cols_monto] *= 10

v0 = crear_variables(panel).set_index(["negocio", "fecha"])[VARIABLES_NUM]
v1 = crear_variables(alterado).set_index(["negocio", "fecha"])[VARIABLES_NUM]
fechas = v0.index.get_level_values("fecha")

hasta_d1 = (v0[fechas <= d + pd.Timedelta(days=1)] - v1[fechas <= d + pd.Timedelta(days=1)]).abs().max()
en_d2 = (v0[fechas == d + pd.Timedelta(days=2)] - v1[fechas == d + pd.Timedelta(days=2)]).abs().max()
pd.DataFrame({"máx. cambio hasta d+1": hasta_d1, "máx. cambio en d+2": en_d2}).round(4)

,máx. cambio hasta d+1,máx. cambio en d+2
lag1,0.00,2.30
lag7,0.00,0.00
lag14,0.00,0.00
media7,0.00,0.33
std7,0.00,0.87
nivel,0.00,0.19
part_efectivo,0.00,0.21
part_datafono,0.00,0.21
part_rumbo,0.00,0.08
part_mi_empresa,0.00,0.04


### Interpretación

Hasta $d+1$ ninguna variable cambia (diferencia exactamente 0): **ninguna usa información del día que pronostica ni de días posteriores**. En $d+2$ cambian las que dependen del día anterior (rezago 1, media, desviación, nivel), como debe ser. Las participaciones también cambian en $d+2$: su ventana de 7 días mezcla días alterados y sin alterar, y eso cambia los pesos. Las que no cambian en $d+2$ también son correctas: el rezago 7 y el 14 de ese día caen antes de $d$, `dias_sin_venta7` solo cuenta ceros, y el festivo no depende de las ventas.

## 🧪 5.4 Experimento: validación aleatoria vs. temporal

Con *k-fold* aleatorio, los días de validación quedan mezclados con días de entrenamiento **del mismo negocio y de la misma semana**, incluso posteriores. Eso le da al modelo información del futuro y hace la validación optimista. Se compara con los folds temporales (cap. 6), usando el mismo modelo.

In [4]:
X, y = train, train["venta_total"]
aleatorio = -cross_val_score(crear_pipeline(C_REF, EPS_REF), X, y, cv=KFold(5, shuffle=True, random_state=SEED), scoring=RMSE_LOG)
temporal = -cross_val_score(crear_pipeline(C_REF, EPS_REF), X, y, cv=folds, scoring=RMSE_LOG)

tabla = pd.DataFrame({
    "RMSE log medio": [aleatorio.mean(), temporal.mean()],
    "desv. entre folds": [aleatorio.std(), temporal.std()],
    "folds": [len(aleatorio), len(temporal)],
}, index=["k-fold aleatorio", "temporal (TimeSeriesSplit + gap)"])
print("RMSE log por fold temporal:", np.round(temporal, 3))
tabla

RMSE log por fold temporal: [1.088 1.053 0.972 0.871]


,RMSE log medio,desv. entre folds,folds
k-fold aleatorio,0.90,0.09,5
temporal (TimeSeriesSplit + gap),1.00,0.08,4


### Interpretación

El k-fold aleatorio da un error **más bajo** que la validación temporal, pero ese número es engañoso: el modelo "vio" días posteriores del mismo negocio. La validación temporal imita el uso real (entrenar con el pasado, pronosticar el futuro) y además muestra que el error **varía entre semanas**, cosa que el k-fold aleatorio esconde. Por eso toda la selección de hiperparámetros del capítulo 8 usa los folds temporales.

## 🧪 5.5 Verificación: el preprocesamiento se ajusta solo con train

El imputador y el escalador están **dentro** del Pipeline, así que en cada fold se ajustan solo con las filas de entrenamiento de ese fold. Se verifica comparando las medias que aprendió el escalador con las medias reales del fold de entrenamiento y de validación.

In [5]:
m = crear_pipeline(C_REF, EPS_REF).fit(train.iloc[i_tr], train["venta_total"].iloc[i_tr])
escalador = m.regressor_.named_steps["pre"].named_transformers_["num"].named_steps["escalar"]
imputador = m.regressor_.named_steps["pre"].named_transformers_["num"].named_steps["imputar"]

ver = pd.DataFrame({
    "media aprendida por el escalador": escalador.mean_,
    "media en train del fold": train.iloc[i_tr][VARIABLES_NUM].fillna(pd.Series(imputador.statistics_, index=VARIABLES_NUM)).mean().values,
    "media en validación del fold": train.iloc[i_val][VARIABLES_NUM].mean().values,
}, index=VARIABLES_NUM)
ver["¿coincide con train?"] = np.isclose(ver.iloc[:, 0], ver.iloc[:, 1], rtol=0, atol=1e-9)
ver["¿coincide con validación?"] = np.isclose(ver.iloc[:, 0], ver.iloc[:, 2], rtol=0, atol=1e-9)
ver.head(8)

,media aprendida por el escalador,media en train del fold,media en validación del fold,¿coincide con train?,¿coincide con validación?
lag1,15.45,15.45,15.45,True,False
lag7,15.46,15.46,15.43,True,False
lag14,15.48,15.48,15.40,True,False
media7,15.45,15.45,15.44,True,False
std7,0.38,0.38,0.33,True,False
nivel,15.47,15.47,15.45,True,False
part_efectivo,0.65,0.65,0.64,True,False
part_datafono,0.26,0.26,0.27,True,False


### Interpretación

Las medias del escalador coinciden exactamente con las del **train del fold** (después de imputar) y no con las de validación: las diferencias son pequeñas pero no nulas, como corresponde a otro periodo. El Pipeline garantiza que la información de validación, y más adelante la de test, no entra en el preprocesamiento.

## ✅ Conclusión de la auditoría

| Fuga | Resultado |
|---|---|
| Medios del mismo día | **Detectada y evitada**: con ellos el error cae casi a la mitad sin que el modelo aprenda nada útil. Solo entran rezagos |
| Variables que miran el futuro | **No hay**: la prueba de perturbación da diferencia 0 hasta $d+1$ |
| Validación barajada | **Evitada**: se usa `TimeSeriesSplit` por días con `gap`. El k-fold aleatorio resultó optimista |
| Preprocesamiento | **Sin fuga**: todo se ajusta dentro del Pipeline |
| Exclusión del negocio con cambio de POS | **Declarada**: se decidió con datos de septiembre, por consistencia de los datos y no por desempeño |

```{warning}
**Qué hacer si el modelo final da "demasiado bien".** Un R² muy alto en este panel **no implica fuga** por sí solo. Gran parte viene de la diferencia de tamaño entre negocios (cap. 4). La prueba real es compararlo con las líneas base ingenuas y mirar el R² **dentro** de cada negocio, lo que se hace en el capítulo 8.
```